In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
import os, sys, gc, time, warnings, pickle, psutil, random, optuna
from sklearn.metrics import mean_squared_error
print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))
print("GPU Details:", tf.config.experimental.list_physical_devices('GPU'))

Num GPUs Available: 1
GPU Details: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [7]:
df = pd.read_parquet('../../data/train_val.parquet')
print(df.head())

def downcast(df):
    cols = df.dtypes.index.tolist()
    types = df.dtypes.values.tolist()
    
    for i, t in enumerate(types):
        col = cols[i]
        if 'int' in str(t):
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)

        elif 'float' in str(t):
            c_min = df[col].min()
            c_max = df[col].max()
            # ⚠️ Không dùng float16 vì không lưu được trong Parquet
            if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)    
    return df

df = downcast(df)
print(df.columns)


                   id  item_id  store_id  state_id  dept_id  cat_id     d  \
0  HOBBIES_1_001_CA_1     1438         1         1        4       2  1029   
1  HOBBIES_1_002_CA_1     1439         1         1        4       2  1029   
2  HOBBIES_1_003_CA_1     1440         1         1        4       2  1029   
3  HOBBIES_1_004_CA_1     1441         1         1        4       2  1029   
4  HOBBIES_1_005_CA_1     1442         1         1        4       2  1029   

   demand  lag_t7  rolling_mean_lag7_w7  ...  event_name_2  snap_CA  snap_TX  \
0     0.0     0.0              1.000000  ...             1        0        0   
1     0.0     1.0              0.142857  ...             1        0        0   
2     0.0     0.0              0.000000  ...             1        0        0   
3     3.0     0.0              1.142857  ...             1        0        0   
4     1.0     1.0              0.428571  ...             1        0        0   

   snap_WI sell_price  disaster_type  disaster_severity_

In [8]:
null_columns = df.columns[df.isna().any()].tolist()

# Kiểm tra giá trị 0
zero_columns = df.columns[(df == 0).any()].tolist()

print("Các cột có giá trị null:", null_columns)
print("Các cột có giá trị 0:", zero_columns)

Các cột có giá trị null: ['demand', 'lag_t7', 'rolling_mean_lag7_w7', 'rolling_mean_lag7_w28', 'sell_price', 'disaster_type']
Các cột có giá trị 0: ['demand', 'lag_t7', 'rolling_mean_lag7_w7', 'rolling_mean_lag7_w28', 'lag_t28', 'rolling_mean_lag28_w7', 'rolling_mean_lag28_w28', 'snap_CA', 'snap_TX', 'snap_WI', 'disaster_severity_flag', 'flag_ufc']


In [ ]:
df = df.drop(['date','disaster_type','disaster_severity_flag', 'flag_ufc', 'humidity'],axis=1)
sample_submission = pd.read_csv('../../data/sample_submission.csv')
sample_submission = downcast(sample_submission)

LAGS = [7, 28]
WINDOWS = [7, 28]
FIRST = 1942 # first to predict
LENGTH = 28

def demand_features_eval(df):
    """ Same as demand_features but for the step-by-step evaluation """
    out = df.groupby('id', sort=False).last()
    for lag in LAGS:
        out[f'lag_t{lag}'] = df.groupby('id', sort=False)['demand'].nth(-lag-1).astype("float32")
        for w in WINDOWS:
            out[f'rolling_mean_lag{lag}_w{w}'] = df.groupby('id', sort=False)['demand'].nth(list(range(-lag-w, -lag))).groupby('id', sort=False).mean().astype("float32")
    
    return out.reset_index()

def prep_train(df,drop_d = 1000):
    
    x = list(set(df.columns) - {'id', 'd', 'demand'})
    print(x)
    # Split into test, valid, train
    test = df[df.d >= FIRST - max(LAGS) - max(WINDOWS) - 28]
    df = df[df.d < FIRST]

    xtrain, xvalid, ytrain, yvalid = train_test_split(df[x], df["demand"], test_size=0.1, shuffle=True, random_state=54)
    train = lgb.Dataset(xtrain, label = ytrain)
    valid = lgb.Dataset(xvalid, label = yvalid)
    return train, valid, test, x

def fit_model(train, valid, num_r):
    """ Fit LightGBM model """
    
    params = {
        'metric': 'rmse',
        'objective': 'poisson',
        'seed': 200,
        'force_row_wise': True,
        'learning_rate': 0.047586309425667044,
        'lambda_l1': 0.03868486588558565,
        'lambda_l2': 8.320083593072901,
        'num_leaves': 2821,
        'feature_fraction': 0.9657558945465752,
        'bagging_fraction': 0.7792310380094438,
        'bagging_freq': 1,
        'min_child_samples': 7,
    }

    callbacks = [
        early_stopping(stopping_rounds=200),
        log_evaluation(period=100)
    ]

    fit = lgb.train(params,
                    train,
                    num_boost_round=num_r,
                    valid_sets=[valid],
                    callbacks=callbacks)

    lgb.plot_importance(fit, importance_type="gain", precision=0, height=0.5, figsize=(6, 10))
    plt.show()

    return fit


def pred_all(fit, test, x):
    """ Calculate predictions """
    
    # Recursive prediction
    for i, day in enumerate(np.arange(FIRST, FIRST + LENGTH)):
        test_day = demand_features_eval(test[(test.d <= day) & (test.d >= day - max(LAGS) - max(WINDOWS))])
        test.loc[test.d == day, "demand"] = fit.predict(test_day[x])
    
    return test

def pred_to_csv(test, cols=sample_submission.columns, file="submission.csv"):
    """ Reshape predictions and save submission csv """
     
    # Prepare for reshaping
    test = test.assign(id=test.id + "_" + np.where(test.d < FIRST, "validation", "evaluation"),
                       F="F" + (test.d - FIRST + LENGTH + 1 - LENGTH * (test.d >= FIRST)).astype("str"))
    
    # Reshape
    submission = test.pivot(index="id", columns="F", values="demand").reset_index()[cols].fillna(1)
    
    # Export
    submission.to_csv(file, index=False)
    
    return True

train, valid, test, x = prep_train(df, 1000 - 28)

del df
gc.collect()

['item_id', 'rolling_mean_lag7_w28', 'event_name_2', 'snap_TX', 'dept_id', 'store_id', 'wday', 'snap_WI', 'snap_CA', 'state_id', 'rolling_mean_lag28_w28', 'cat_id', 'month', 'rolling_mean_lag28_w7', 'year', 'event_name_1', 'sell_price', 'lag_t7', 'lag_t28', 'rolling_mean_lag7_w7', 'gold_price']


172

In [ ]:
fit = fit_model(train, valid,200)
pred = pred_all(fit, test, x)
pred_to_csv(pred, cols=sample_submission.columns, file="../submission.csv")

[LightGBM] [Warning] feature_fraction is set=0.9657558945465752, colsample_bytree=0.7 will be ignored. Current value: feature_fraction=0.9657558945465752
[LightGBM] [Warning] bagging_fraction is set=0.7792310380094438, sub_row=0.7 will be ignored. Current value: bagging_fraction=0.7792310380094438
[LightGBM] [Warning] feature_fraction is set=0.9657558945465752, colsample_bytree=0.7 will be ignored. Current value: feature_fraction=0.9657558945465752
[LightGBM] [Warning] bagging_fraction is set=0.7792310380094438, sub_row=0.7 will be ignored. Current value: bagging_fraction=0.7792310380094438
[LightGBM] [Info] Total Bins 2034
[LightGBM] [Info] Number of data points in the train set: 25053633, number of used features: 21
[LightGBM] [Warning] feature_fraction is set=0.9657558945465752, colsample_bytree=0.7 will be ignored. Current value: feature_fraction=0.9657558945465752
[LightGBM] [Warning] bagging_fraction is set=0.7792310380094438, sub_row=0.7 will be ignored. Current value: bagging_f